# 08 LR on the full training set: main runs, PCA vs LDA, stacking

All LR runs for the report's results table, trained on **all 1.6M training domains** and
scored on the **validation set**.

| Part | What |
|---|---|
| 1 | LR on all features, binary vs family-first, with and without TLD; PCA and LDA at 95% and 99% |
| 2 | Plots: PCA vs LDA |
| 3 | Stacking, without TLD |
| 3b | Stacking, with TLD (optional) |
| 4 | Table rows for report.tex |

Each part saves its results to `results/full_val/`. A part whose results file already
exists **loads it instead of retraining**, so the notebook can be re-run from the top in a
few minutes.

## Settings

- `N_JOBS`: fits running at the same time, about 3 GB of memory each: 4 for 16 GB of RAM, 2 for 8 GB.
- `PCA_LEVELS`: shares of variance kept by PCA; LDA gets the same numbers of components.
- `RUN_STACK_TLD`: `True` to run Part 3b (stacking with TLD, about 1 hour).

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

EVAL_SET = "val"
N_JOBS = 4
PCA_LEVELS = [0.95, 0.99]
RUN_STACK_TLD = False
OUT_DIR = ROOT / "results" / "full_val"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Data and features

The team's split and the 74 cached features.

In [ ]:
import numpy as np
import pandas as pd

from src.data.dataset import load_splits
from src.features.build import build_features

train_full, val, test = load_splits()
feats = build_features(train_full, val, test)
name_cols, tld_cols = feats["name_cols"], feats["tld_cols"]
evaluate, X_eval_all = (test, feats["test"]) if EVAL_SET == "test" else (val, feats["val"])
print(f"train {len(train_full):,} | {EVAL_SET} {len(evaluate):,} | "
      f"{len(name_cols)} name + {len(tld_cols)} TLD features")

## Part 1: all features, PCA and LDA

How many components each PCA level keeps (PCA fitted on the standardized training
features). LDA gets the **same numbers of components**, capped at classes − 1 (so binary
LDA has 1 direction).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

counts = {}
for tld, cols in [("yes", name_cols + tld_cols), ("no", name_cols)]:
    Xs = StandardScaler().fit_transform(feats["train"][cols])
    cum = np.cumsum(PCA().fit(Xs).explained_variance_ratio_)
    counts[tld] = {lvl: int(np.searchsorted(cum, lvl) + 1) for lvl in PCA_LEVELS}
del Xs
pd.DataFrame(counts).rename_axis("variance kept").rename(columns={"yes": "with TLD", "no": "without TLD"})

All features and every PCA and LDA variant, binary and family-first, with and without TLD,
in one parallel batch. Loaded from `logreg_full_val_results.csv` if it exists.

In [ ]:
import pandas as pd
from src.models.experiment import run_variants

batch_file = OUT_DIR / "logreg_full_val_results.csv"
if batch_file.exists():
    batch = pd.read_csv(batch_file, index_col=[0, 1, 2])
    print("part 1 loaded from", batch_file.name)
else:
    variants = {"all features": {}}
    for lvl in PCA_LEVELS:
        variants[f"PCA {lvl:.0%}"] = {"yes": {"pca": counts["yes"][lvl]}, "no": {"pca": counts["no"][lvl]}}
        variants[f"LDA {lvl:.0%}"] = {"yes": {"lda": counts["yes"][lvl]}, "no": {"lda": counts["no"][lvl]}}
    t0 = time.time()
    batch, batch_recalls = run_variants(feats, train_full, evaluate, X_eval_all, variants,
                                        n_jobs=N_JOBS, run_name="logreg_full_val",
                                        eval_set=EVAL_SET, save_dir=OUT_DIR)
    print(f"part 1 done in {(time.time() - t0) / 60:.0f} min")

Main rows (all features) with every metric, and the convergence check.

In [ ]:
main = batch.loc["all features"]
main[["macro_f1", "dga_precision", "dga_recall", "roc_auc", "iterations", "converged", "fit_s"]]

Macro-F1 and number of components for every PCA and LDA variant.

In [ ]:
batch[["macro_f1", "n_components"]].astype(float).unstack(["setup", "TLD"]).round(4)

## Part 2: plots

**Figure A**: macro-F1 against the number of components, PCA vs LDA, family-first
(solid) and binary (dashed); grey lines: all features, no reduction.

In [ ]:
import matplotlib.pyplot as plt

tab = batch[["macro_f1", "n_components"]].astype(float)
allf = tab.loc["all features", "macro_f1"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, tld, title in [(axes[0], "yes", "with TLD"), (axes[1], "no", "without TLD")]:
    for method, color in [("PCA", "tab:blue"), ("LDA", "tab:orange")]:
        for setup, style in [("family", "-"), ("binary", "--")]:
            rows = [tab.loc[(f"{method} {l:.0%}", setup, tld)] for l in PCA_LEVELS]
            ax.plot([r["n_components"] for r in rows], [r["macro_f1"] for r in rows],
                    style, marker="o", color=color, label=f"{method}, {setup}")
    for setup, style in [("family", "-"), ("binary", "--")]:
        ax.axhline(allf[setup, tld], color="grey", ls=style, lw=1, label=f"all features, {setup}")
    ax.set_title(f"LR, {title}")
    ax.set_xlabel("number of components")
axes[0].set_ylabel("macro-F1 (validation)")
axes[0].legend(fontsize=8, loc="lower center")
plt.tight_layout()
plt.savefig(OUT_DIR / "pca_vs_lda.png", dpi=200)
plt.show()

**Figure B**: validation domains on the first two PCA and LDA directions (LDA fitted
with the family labels), name features only. The TLD is left out because with it LDA's
first directions only isolate a few single-TLD families. Axes are cut to the 1st–99th
percentile. A picture only: the models use 10–62 dimensions, not 2.

In [ ]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

cols = name_cols
fit_rows = train_full.sample(n=200_000, random_state=0)
scaler = StandardScaler().fit(feats["train"].loc[fit_rows.index, cols])
Xfit = scaler.transform(feats["train"].loc[fit_rows.index, cols])
pca2 = PCA(n_components=2).fit(Xfit)
lda2 = LinearDiscriminantAnalysis(n_components=2).fit(Xfit, fit_rows["family"])

show = evaluate.sample(n=6000, random_state=0)
Xshow = scaler.transform(X_eval_all.loc[show.index, cols])
is_dga = (show["label"] == "dga").to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, proj, title in [(axes[0], pca2.transform(Xshow), "PCA (ignores labels)"),
                        (axes[1], lda2.transform(Xshow), "LDA (uses family labels)")]:
    ax.scatter(*proj[~is_dga].T, s=3, alpha=0.4, label="legitimate")
    ax.scatter(*proj[is_dga].T, s=3, alpha=0.4, label="DGA")
    lo, hi = np.percentile(proj, [1, 99], axis=0)
    pad = 0.1 * (hi - lo)
    ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0]); ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
    ax.set_title(title)
    ax.set_xlabel("component 1"); ax.set_ylabel("component 2")
axes[0].legend(markerscale=4)
plt.suptitle("Name features only (no TLD), 6000 validation domains")
plt.tight_layout()
plt.savefig(OUT_DIR / "pca_lda_2d.png", dpi=200)
plt.show()

## Part 3: stacking without TLD

The family-first probabilities, out-of-fold over 5 folds, become the input of a binary LR
(6 family-first fits: 5 folds + 1 full). Loaded from its results file if it exists.

In [ ]:
from src.models.stacking import run_stacking

def stacking(tld, run_name):
    path = OUT_DIR / f"{run_name}_results.csv"
    if path.exists():
        print("loaded from", path.name)
        return pd.read_csv(path, index_col=[0, 1, 2], dtype={"TLD": str})
    t0 = time.time()
    table = run_stacking(feats, train_full, evaluate, X_eval_all, seeds=[0], sample=None,
                         tlds=(tld,), n_jobs=N_JOBS, run_name=run_name,
                         eval_set=EVAL_SET, save_dir=OUT_DIR)
    print(f"done in {(time.time() - t0) / 60:.0f} min")
    return table

stack = stacking("no", "logreg_stacking_full_val")
stack.droplevel("seed")[["macro_f1", "dga_precision", "dga_recall", "roc_auc"]].astype(float).round(4)

## Part 3b: stacking with TLD (optional)

Same as Part 3 on all 74 features. Runs only if `RUN_STACK_TLD = True` (about 1 hour), or
loads its results file if it exists. Saved under its own name, so Part 3 is kept.

In [ ]:
stack_tld = None
if RUN_STACK_TLD or (OUT_DIR / "logreg_stacking_full_val_tld_results.csv").exists():
    stack_tld = stacking("yes", "logreg_stacking_full_val_tld")
    display(stack_tld.droplevel("seed")[["macro_f1", "dga_precision", "dga_recall", "roc_auc"]]
            .astype(float).round(4))
else:
    print("skipped (set RUN_STACK_TLD = True to run it)")

## Part 4: table rows for report.tex

The LR rows of the results table, in the same order and with the same labels as the
report. "Kept epoch" does not apply to LR (--).

In [ ]:
M = ["macro_f1", "dga_precision", "dga_recall", "roc_auc"]
def row(label, setup, v):
    return f"{label} & {setup} & " + " & ".join(f"{x:.4f}" for x in v) + " & -- \\\\"
def delta(v):
    return " & $\\Delta$ & " + " & ".join(f"${x:+.4f}$" for x in v) + " & \\\\"

b = batch[M].astype(float)
m = b.loc["all features"]
lines = []
for label, tld in [("Model 1 (LR), with TLD", "yes"), ("LR, without TLD", "no")]:
    lines += [row(label, "binary", m.loc["binary", tld]),
              row("", "family-first", m.loc["family", tld]),
              delta(m.loc["family", tld] - m.loc["binary", tld])]
lines.append("\\midrule")
if stack_tld is not None:
    lines.append(row("LR + stacking, with TLD", "binary",
                     stack_tld.droplevel("seed")[M].astype(float).loc[("yes", "stack_probs")]))
lines.append(row("LR + stacking, without TLD", "binary",
                 stack.droplevel("seed")[M].astype(float).loc[("no", "stack_probs")]))
for method in ["PCA", "LDA"]:
    lvl = f"{method} 95%"
    for tld, name in [("yes", "with TLD"), ("no", "without TLD")]:
        lines.append(row(f"LR + {method} 95\\%, {name}", "family-first", b.loc[(lvl, "family", tld)]))
print("\n".join(lines))

**Result.**
- **Family-first wins** for LR: +0.0185 macro-F1 with the TLD, +0.0061 without.
- **Stacking** (without TLD) beats binary by +0.0050 but stays 0.0011 below plain family-first.
- **PCA** lowers macro-F1 at both levels, because it ignores the labels.
- **LDA** with the family labels keeps the information: family-first with 55 dimensions
  (0.9409) is level with all 74 features (0.9397). Binary LDA keeps only one direction and drops.